
# ARC-v0.18 — Cross-Encoder FEVER Replication

**Goal:** test whether the approximation-feedback findings obtained with `BAAI/bge-small-en-v1.5` replicate under a second encoder family **without selecting the new encoder or policy settings after seeing outcomes**.

This notebook opens a new paper-facing generalization line and does not modify the sealed ARC-v0.13–v0.17.1 evidence.

## Frozen second encoder

Default: `intfloat/e5-small-v2` (384 dimensions), with E5 prefixes `query:` and `passage:` and L2-normalized embeddings. This keeps a genuinely different encoder family while making a 5.4M-document Colab replication feasible.

## Frozen questions

1. Do aggregate H1/H2/H3 directions remain positive?
2. Is FEVER still dominated by stable/null trajectories?
3. Is amplification still a minority regime under the same threshold family?
4. Does stronger feedback gain still increase amplification risk?
5. Does configuration-level risk transfer from FIT to untouched validation?

## Frozen retrieval / feedback design

- IVF-PQ32: `nlist=4096`, `m=32`, `nbits=8`
- IVF-SQ8: `nlist=4096`, 8-bit scalar quantization
- `nprobe=64`
- top-100 retrieval, nDCG@10 utility
- four feedback updates
- same 44 mean/softmax configurations
- same authoritative FEVER FIT/validation membership recovered from sealed ARC-v0.13 checkpoints

The higher-fidelity SQ8 condition is a relative comparator, not an exact oracle.

## Claim gate

- `REPLICATION`: positive validation H1/H2/H3 means, stable/null majority, amplification minority, and alpha=0.7 amplification > alpha=0.1.
- `PARTIAL_REPLICATION`: at least half but not all criteria hold.
- `NON_REPLICATION`: fewer than half hold.

Negative results are retained. Do not switch encoders after outcomes are observed.


In [ ]:

%pip install -q sentence-transformers==3.4.1 faiss-cpu==1.12.0 pyarrow pandas numpy scipy scikit-learn tqdm requests


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, random, time, warnings, zipfile

import faiss
import numpy as np
import pandas as pd
import requests
from scipy.stats import pearsonr, spearmanr
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260819
random.seed(SEED); np.random.seed(SEED)

ENCODER_NAME = "intfloat/e5-small-v2"
ENCODER_TAG = "e5-small-v2"
DIM = 384
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

N_DOCS = 5_416_568
N_DEV = 6_666
N_FIT = 3_350
N_VAL = 3_316

NLIST = 4096
NPROBE = 64
PQ_M = 32
PQ_NBITS = 8
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4
EPS_PRIMARY = 0.002
EPS_SWEEP = [0.0, 0.001, 0.002, 0.005, 0.01]

ENCODE_BATCH = 512
CORPUS_SHARD_SIZE = 100_000
INDEX_ADD_BATCH = 100_000
TRAIN_SAMPLE = 500_000

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir(): drive.mount("/content/drive")
assert DRIVE_ROOT.is_dir()
ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
ARC_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive:", DRIVE_ROOT)
print("Encoder:", ENCODER_NAME)


In [ ]:

# Recover the exact sealed ARC-v0.13 membership instead of guessing the split function.
V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"
preferred = V013_ROOT / "20260817-140640"

def complete_v013(p):
    return p.is_dir() and len(list(p.glob("fit-*.parquet"))) == 44 and len(list(p.glob("validation-*.parquet"))) == 44

if complete_v013(preferred):
    V013_RUN = preferred
else:
    candidates = sorted([p for p in V013_ROOT.iterdir() if complete_v013(p)], reverse=True)
    assert candidates, "No complete ARC-v0.13 run found"
    V013_RUN = candidates[0]

fit_files = sorted(V013_RUN.glob("fit-*.parquet"))
val_files = sorted(V013_RUN.glob("validation-*.parquet"))

def qset(p):
    return set(pd.read_parquet(p, columns=["query_id"])["query_id"].astype(str))

FIT_IDS = qset(fit_files[0]); VAL_IDS = qset(val_files[0])
assert len(FIT_IDS)==N_FIT and len(VAL_IDS)==N_VAL and not (FIT_IDS & VAL_IDS)
for p in fit_files[1:]: assert qset(p)==FIT_IDS
for p in val_files[1:]: assert qset(p)==VAL_IDS
DEV_IDS = FIT_IDS | VAL_IDS
assert len(DEV_IDS)==N_DEV

def membership_sha(ids):
    return hashlib.sha256("\n".join(sorted(ids)).encode()).hexdigest()

FIT_SHA = membership_sha(FIT_IDS); VAL_SHA = membership_sha(VAL_IDS)
print("v0.13 source:", V013_RUN)
print("FIT/VAL:", len(FIT_IDS), len(VAL_IDS))
print("FIT SHA:", FIT_SHA)
print("VAL SHA:", VAL_SHA)
print("FROZEN SPLIT RECOVERY — PASS")



## Raw FEVER text

The second encoder must be built from raw text. The notebook first uses a Drive-local BEIR FEVER copy; if absent it downloads the public BEIR FEVER archive into Drive. Only the frozen DEV membership is evaluated; FEVER test outcomes are not used.


In [ ]:

RAW_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "raw-datasets"
RAW_ROOT.mkdir(parents=True, exist_ok=True)
FEVER_DIR = RAW_ROOT / "fever"
CORPUS_JSONL = FEVER_DIR / "corpus.jsonl"
QUERIES_JSONL = FEVER_DIR / "queries.jsonl"

if not (CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file()):
    zip_path = RAW_ROOT / "fever.zip"
    if not zip_path.is_file():
        url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fever.zip"
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length",0))
            with open(zip_path,"wb") as f, tqdm(total=total,unit="B",unit_scale=True,desc="fever.zip") as bar:
                for chunk in r.iter_content(8*1024*1024):
                    if chunk: f.write(chunk); bar.update(len(chunk))
    with zipfile.ZipFile(zip_path,"r") as zf: zf.extractall(RAW_ROOT)

assert CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file()
print("RAW FEVER — READY")


In [ ]:

def iter_jsonl(path):
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            if line.strip(): yield json.loads(line)

query_text = {str(o["_id"]): str(o.get("text", "")) for o in iter_jsonl(QUERIES_JSONL)}
missing = sorted(DEV_IDS - set(query_text))
assert not missing, f"Missing frozen DEV query texts, e.g. {missing[:10]}"
DEV_QUERY_IDS = sorted(DEV_IDS)
DEV_QUERY_INDEX = {qid:i for i,qid in enumerate(DEV_QUERY_IDS)}
print("Raw query pool:", len(query_text))
print("Frozen DEV queries:", len(DEV_QUERY_IDS))


In [ ]:

# Seal protocol before any E5 retrieval outcomes are inspected.
V018_ROOT = ARC_ROOT / "cross-encoder-fever-replication-v018"
V018_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V018_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PROTOCOL = {
 "status":"ARC_V018_CROSS_ENCODER_PROTOCOL_SEALED_BEFORE_OUTCOMES",
 "created_at_utc":datetime.now(timezone.utc).isoformat(),
 "source_v013_run":str(V013_RUN),
 "fit_membership_sha256":FIT_SHA,
 "validation_membership_sha256":VAL_SHA,
 "encoder":ENCODER_NAME,"dimension":DIM,"query_prefix":QUERY_PREFIX,"passage_prefix":PASSAGE_PREFIX,
 "retrievers":{
   "PQ32":{"type":"IndexIVFPQ","nlist":NLIST,"m":PQ_M,"nbits":PQ_NBITS,"nprobe":NPROBE},
   "SQ8":{"type":"IndexIVFScalarQuantizer","nlist":NLIST,"quantizer":"QT_8bit","nprobe":NPROBE}},
 "feedback":{"rounds":MAX_ROUNDS,"top_retrieve":TOP_RETRIEVE,"utility_k":TOP_K,
             "alphas":[0.1,0.3,0.5,0.7],"mean_k":[5,20,50],"softmax_k":[5,20],
             "softmax_temperatures":[0.05,0.1,0.2,0.5]},
 "endpoints":["H1_query_state_slope","H2_candidate_jaccard_slope","H3_abs_ndcg10_gap_slope"],
 "epsilon_primary":EPS_PRIMARY,"epsilon_sweep":EPS_SWEEP,
 "test_accessed":False,"test_relevance_accessed":False
}
PROTOCOL_PATH = OUT / "v018_cross_encoder_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL,indent=2,sort_keys=True),encoding="utf-8")

def sha256_file(path, chunk=16*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)
(OUT/"V018_PROTOCOL_SHA256.txt").write_text(f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n")
print("Output:", OUT)
print("Protocol SHA:", PROTOCOL_SHA)
print("PROTOCOL SEALED — PASS")


In [ ]:

model = SentenceTransformer(ENCODER_NAME)
QUERY_EMB_PATH = OUT / "dev_query_embeddings.float32.npy"
QUERY_IDS_PATH = OUT / "dev_query_ids.txt"

if QUERY_EMB_PATH.is_file():
    dev_query_embeddings = np.load(QUERY_EMB_PATH)
    assert QUERY_IDS_PATH.read_text().splitlines()==DEV_QUERY_IDS
else:
    texts=[QUERY_PREFIX+query_text[qid] for qid in DEV_QUERY_IDS]
    dev_query_embeddings=model.encode(texts,batch_size=ENCODE_BATCH,convert_to_numpy=True,
                                      normalize_embeddings=True,show_progress_bar=True).astype(np.float32)
    np.save(QUERY_EMB_PATH,dev_query_embeddings)
    QUERY_IDS_PATH.write_text("\n".join(DEV_QUERY_IDS)+"\n")
assert dev_query_embeddings.shape==(N_DEV,DIM)
print("QUERY ENCODING — PASS", dev_query_embeddings.shape)


In [ ]:

# Encode corpus in resumable shards.
SHARD_DIR = OUT / "corpus_shards"; SHARD_DIR.mkdir(exist_ok=True)
shard_records=[]; ids_buf=[]; text_buf=[]; shard_idx=0; total_docs=0

def persist_shard(idx, ids, texts):
    ep=SHARD_DIR/f"shard-{idx:04d}.float16.npy"; ip=SHARD_DIR/f"shard-{idx:04d}.ids.txt"
    if ep.is_file() and ip.is_file():
        saved=ip.read_text().splitlines(); arr=np.load(ep,mmap_mode="r")
        assert saved==ids and arr.shape==(len(ids),DIM)
        return {"shard":idx,"rows":len(ids),"embedding_path":str(ep),"ids_path":str(ip),"sha256":sha256_file(ep),"reused":True}
    vec=model.encode([PASSAGE_PREFIX+t for t in texts],batch_size=ENCODE_BATCH,convert_to_numpy=True,
                     normalize_embeddings=True,show_progress_bar=True).astype(np.float16)
    np.save(ep,vec); ip.write_text("\n".join(ids)+"\n")
    return {"shard":idx,"rows":len(ids),"embedding_path":str(ep),"ids_path":str(ip),"sha256":sha256_file(ep),"reused":False}

for o in tqdm(iter_jsonl(CORPUS_JSONL),total=N_DOCS,desc="FEVER corpus"):
    did=str(o["_id"]); title=str(o.get("title","") or ""); body=str(o.get("text","") or "")
    ids_buf.append(did); text_buf.append((title+" "+body).strip()); total_docs+=1
    if len(ids_buf)>=CORPUS_SHARD_SIZE:
        rec=persist_shard(shard_idx,ids_buf,text_buf); shard_records.append(rec); print(rec)
        shard_idx+=1; ids_buf=[]; text_buf=[]
if ids_buf:
    rec=persist_shard(shard_idx,ids_buf,text_buf); shard_records.append(rec); print(rec)
assert total_docs==N_DOCS
(OUT/"corpus_encoding_manifest.json").write_text(json.dumps({"status":"COMPLETE","rows":total_docs,"shards":shard_records},indent=2))
print("CORPUS ENCODING — PASS", len(shard_records), "shards")


In [ ]:

DOC_IDS_PATH=OUT/"corpus_doc_ids.txt"
if DOC_IDS_PATH.is_file(): DOC_IDS=DOC_IDS_PATH.read_text().splitlines()
else:
    DOC_IDS=[]
    for rec in shard_records: DOC_IDS.extend(Path(rec["ids_path"]).read_text().splitlines())
    assert len(DOC_IDS)==N_DOCS
    DOC_IDS_PATH.write_text("\n".join(DOC_IDS)+"\n")
DOC_ID_TO_ROW={d:i for i,d in enumerate(DOC_IDS)}
assert len(DOC_ID_TO_ROW)==N_DOCS
print("DOC MAPPING — PASS")


In [ ]:

CORPUS_MEMMAP_PATH=OUT/"corpus_embeddings.float16.memmap"
expected=N_DOCS*DIM*np.dtype(np.float16).itemsize
if not (CORPUS_MEMMAP_PATH.is_file() and CORPUS_MEMMAP_PATH.stat().st_size==expected):
    mm=np.memmap(CORPUS_MEMMAP_PATH,dtype=np.float16,mode="w+",shape=(N_DOCS,DIM)); cur=0
    for rec in tqdm(shard_records,desc="Merge shards"):
        a=np.load(rec["embedding_path"],mmap_mode="r"); n=len(a); mm[cur:cur+n]=a; cur+=n
    assert cur==N_DOCS; mm.flush(); del mm; gc.collect()
corpus_mm=np.memmap(CORPUS_MEMMAP_PATH,dtype=np.float16,mode="r",shape=(N_DOCS,DIM))
probe=np.asarray(corpus_mm[np.linspace(0,N_DOCS-1,2000,dtype=np.int64)],dtype=np.float32)
norms=np.linalg.norm(probe,axis=1)
assert np.isfinite(probe).all() and np.all((norms>0.9)&(norms<1.1))
print("MEMMAP — PASS", CORPUS_MEMMAP_PATH)


In [ ]:

TRAIN_PATH=OUT/"index_train_sample.float32.npy"
if TRAIN_PATH.is_file(): train_x=np.load(TRAIN_PATH)
else:
    rr=np.random.default_rng(SEED); rows=np.sort(rr.choice(N_DOCS,size=min(TRAIN_SAMPLE,N_DOCS),replace=False))
    train_x=np.asarray(corpus_mm[rows],dtype=np.float32); np.save(TRAIN_PATH,train_x)
print("INDEX TRAIN SAMPLE", train_x.shape)


In [ ]:

PQ32_PATH=OUT/f"fever-{ENCODER_TAG}-ivfpq-nlist{NLIST}-m{PQ_M}-nbits{PQ_NBITS}.faiss"
SQ8_PATH=OUT/f"fever-{ENCODER_TAG}-ivfsq8-nlist{NLIST}.faiss"

def add_all(index,path):
    start=int(index.ntotal)
    for left in tqdm(range(start,N_DOCS,INDEX_ADD_BATCH),desc=path.name):
        right=min(left+INDEX_ADD_BATCH,N_DOCS); index.add(np.asarray(corpus_mm[left:right],dtype=np.float32)); faiss.write_index(index,str(path))
    assert index.ntotal==N_DOCS; return index

if PQ32_PATH.is_file(): pq32=faiss.read_index(str(PQ32_PATH))
else:
    pq32=faiss.IndexIVFPQ(faiss.IndexFlatIP(DIM),DIM,NLIST,PQ_M,PQ_NBITS,faiss.METRIC_INNER_PRODUCT)
    pq32.train(train_x); faiss.write_index(pq32,str(PQ32_PATH))
pq32=add_all(pq32,PQ32_PATH); pq32.nprobe=NPROBE

if SQ8_PATH.is_file(): sq8=faiss.read_index(str(SQ8_PATH))
else:
    sq8=faiss.IndexIVFScalarQuantizer(faiss.IndexFlatIP(DIM),DIM,NLIST,faiss.ScalarQuantizer.QT_8bit,faiss.METRIC_INNER_PRODUCT)
    sq8.train(train_x); faiss.write_index(sq8,str(SQ8_PATH))
sq8=add_all(sq8,SQ8_PATH); sq8.nprobe=NPROBE
print("INDEX BUILD — PASS", pq32.ntotal, sq8.ntotal)


In [ ]:

# Prefer the already-audited local DEV qrels in corpus-row space.
SEALED_QRELS=DRIVE_ROOT/"hc-rars-fever-5m-untouched-confirmation-v1"/"stage2"/"dev_qrels_rows.csv"
assert SEALED_QRELS.is_file(), f"Missing audited DEV qrels: {SEALED_QRELS}"
qr=pd.read_csv(SEALED_QRELS)
assert {"query-id","corpus-row","score"}.issubset(qr.columns)
QRELS=defaultdict(set)
for _,r in qr.iterrows():
    qid=str(r["query-id"])
    if qid in DEV_IDS and float(r["score"])>0: QRELS[qid].add(int(r["corpus-row"]))
missing=[q for q in DEV_QUERY_IDS if not QRELS[q]]
assert not missing, f"Missing qrels for {len(missing)} DEV queries"
print("DEV QRELS — PASS")


In [ ]:

def norm_vec(x):
    x=np.asarray(x,dtype=np.float32); return x/max(float(np.linalg.norm(x)),1e-12)

def fetch_docs(rows): return np.asarray(corpus_mm[np.asarray(rows,dtype=np.int64)],dtype=np.float32)

def retrieve(index,q,k=TOP_RETRIEVE):
    s,i=index.search(norm_vec(q).reshape(1,-1),k); return s[0],i[0]

def ndcg(ids,rel,k=TOP_K):
    if not rel: return 0.0
    gains=np.array([1.0 if int(i) in rel else 0.0 for i in ids[:k]])
    dcg=float((gains/np.log2(np.arange(2,len(gains)+2))).sum())
    m=min(k,len(rel)); idcg=float((np.ones(m)/np.log2(np.arange(2,m+2))).sum())
    return dcg/idcg if idcg else 0.0

def recall10(ids,rel): return len(set(map(int,ids[:10]))&rel)/len(rel) if rel else 0.0

def mrr10(ids,rel):
    for r,d in enumerate(ids[:10],1):
        if int(d) in rel: return 1.0/r
    return 0.0

one=[]
for qid in tqdm(DEV_QUERY_IDS,desc="one-shot"):
    q=dev_query_embeddings[DEV_QUERY_INDEX[qid]]; rel=QRELS[qid]
    for name,index in [("PQ32",pq32),("SQ8",sq8)]:
        _,ids=retrieve(index,q); one.append({"query_id":qid,"retriever":name,"recall10":recall10(ids,rel),"mrr10":mrr10(ids,rel),"ndcg10":ndcg(ids,rel)})
one=pd.DataFrame(one)
summary=one.groupby("retriever",as_index=False).agg(recall10=("recall10","mean"),mrr10=("mrr10","mean"),ndcg10=("ndcg10","mean"))
display(summary)
one.to_parquet(OUT/"v018_one_shot_query_metrics.parquet",index=False); summary.to_csv(OUT/"v018_one_shot_summary.csv",index=False)
if summary.set_index("retriever").loc["SQ8","ndcg10"] < summary.set_index("retriever").loc["PQ32","ndcg10"]:
    raise RuntimeError("SQ8 is not higher-fidelity by DEV nDCG@10 under E5. Stop; do not silently relabel conditions.")
print("ONE-SHOT FIDELITY AUDIT — PASS")


In [ ]:

CONFIGS=[]
for a in [0.1,0.3,0.5,0.7]:
    for k in [5,20,50]: CONFIGS.append({"method":"mean","alpha":a,"k":k,"temperature":None,"config_key":f"mean-k{k}-a{str(a).replace('.','p')}-tnone"})
for a in [0.1,0.3,0.5,0.7]:
    for k in [5,20]:
        for t in [0.05,0.1,0.2,0.5]: CONFIGS.append({"method":"softmax","alpha":a,"k":k,"temperature":t,"config_key":f"softmax-k{k}-a{str(a).replace('.','p')}-t{str(t).replace('.','p')}"})
assert len(CONFIGS)==44
print("CONFIGS",len(CONFIGS))


In [ ]:

def feedback(index,q,cfg):
    scores,ids=retrieve(index,q,TOP_RETRIEVE); k=int(cfg["k"]); docs=fetch_docs(ids[:k]); docs/=np.maximum(np.linalg.norm(docs,axis=1,keepdims=True),1e-12)
    if cfg["method"]=="mean": f=docs.mean(axis=0)
    else:
        z=scores[:k].astype(np.float64)/float(cfg["temperature"]); z-=z.max(); w=np.exp(z); w/=w.sum(); f=(docs*w[:,None]).sum(axis=0)
    return norm_vec(f)

def update(q0,fb,a): return norm_vec((1-float(a))*q0+float(a)*fb)

def jacdist(a,b):
    A=set(map(int,a)); B=set(map(int,b)); return 1-len(A&B)/len(A|B) if (A or B) else 0.0

def run_pair(qid,cfg):
    q0=dev_query_embeddings[DEV_QUERY_INDEX[qid]]; qL=q0.copy(); qH=q0.copy(); rel=QRELS[qid]; rows=[]
    for t in range(MAX_ROUNDS+1):
        _,iL=retrieve(pq32,qL); _,iH=retrieve(sq8,qH); uL=ndcg(iL,rel); uH=ndcg(iH,rel)
        rows.append({"query_id":qid,"iteration":t,"method":cfg["method"],"alpha":cfg["alpha"],"k":cfg["k"],"temperature":cfg["temperature"],"config_key":cfg["config_key"],
                     "query_state_distance":1-float(np.dot(norm_vec(qL),norm_vec(qH))),"candidate_jaccard_distance":jacdist(iL[:TOP_RETRIEVE],iH[:TOP_RETRIEVE]),
                     "utility_low":uL,"utility_high":uH,"signed_utility_gap":uH-uL,"abs_utility_gap":abs(uH-uL)})
        if t==MAX_ROUNDS: break
        qL=update(q0,feedback(pq32,qL,cfg),cfg["alpha"]); qH=update(q0,feedback(sq8,qH,cfg),cfg["alpha"])
    return rows

def slope(x,y): return float(np.polyfit(np.asarray(x,float),np.asarray(y,float),1)[0])

def endpoint_df(traj):
    out=[]
    for keys,g in traj.groupby(["query_id","method","alpha","k","temperature","config_key"],dropna=False,sort=False):
        g=g.sort_values("iteration"); x=g["iteration"].to_numpy(float)
        out.append({"query_id":keys[0],"method":keys[1],"alpha":keys[2],"k":keys[3],"temperature":keys[4],"config_key":keys[5],
                    "H1_slope":slope(x,g["query_state_distance"]),"H2_slope":slope(x,g["candidate_jaccard_distance"]),
                    "H3_abs_slope":slope(x,g["abs_utility_gap"]),"H3_signed_slope":slope(x,g["signed_utility_gap"]),"final_signed_gap":float(g["signed_utility_gap"].iloc[-1])})
    return pd.DataFrame(out)
print("TRAJECTORY FUNCTIONS — READY")


In [ ]:

# Smoke test
sm=[]
for qid in sorted(FIT_IDS)[:5]: sm.extend(run_pair(qid,CONFIGS[0]))
sm=pd.DataFrame(sm); assert len(sm)==5*(MAX_ROUNDS+1); assert np.isfinite(sm.select_dtypes(include=[np.number])).all().all()
display(sm.head())
print("SMOKE TEST — PASS")


In [ ]:

# Full FIT and untouched-validation sweeps. Each config is independently checkpointed.
FIT_DIR=OUT/"fit"; VAL_DIR=OUT/"validation"; FIT_DIR.mkdir(exist_ok=True); VAL_DIR.mkdir(exist_ok=True)

def run_sweep(ids,directory,prefix):
    ids=sorted(ids)
    for j,cfg in enumerate(CONFIGS,1):
        p=directory/f"{prefix}-{cfg['config_key']}.parquet"
        if p.is_file():
            d=pd.read_parquet(p); assert d["query_id"].nunique()==len(ids); print(f"[{prefix.upper()} {j:02d}/44] REUSE",p.name); continue
        print("\n"+"="*90); print(f"[{prefix.upper()} {j:02d}/44]",cfg["config_key"]); print("="*90)
        rows=[]; t0=time.perf_counter()
        for qid in tqdm(ids,desc=cfg["config_key"]): rows.extend(run_pair(qid,cfg))
        d=pd.DataFrame(rows); assert d["query_id"].nunique()==len(ids) and len(d)==len(ids)*(MAX_ROUNDS+1)
        d.to_parquet(p,index=False); print("SAVED",p.name,"seconds",time.perf_counter()-t0,"sha256",sha256_file(p))

run_sweep(FIT_IDS,FIT_DIR,"fit")
print("FIT SWEEP — COMPLETE")
run_sweep(VAL_IDS,VAL_DIR,"validation")
print("VALIDATION SWEEP — COMPLETE")


In [ ]:

def load_endpoints(directory,prefix):
    frames=[]
    for p in sorted(directory.glob(f"{prefix}-*.parquet")): frames.append(endpoint_df(pd.read_parquet(p)))
    return pd.concat(frames,ignore_index=True)

fit_ep=load_endpoints(FIT_DIR,"fit"); val_ep=load_endpoints(VAL_DIR,"validation")
assert fit_ep["config_key"].nunique()==44 and val_ep["config_key"].nunique()==44
fit_ep.to_parquet(OUT/"v018_fit_endpoints.parquet",index=False); val_ep.to_parquet(OUT/"v018_validation_endpoints.parquet",index=False)
print("ENDPOINTS",fit_ep.shape,val_ep.shape)


In [ ]:

aggregate=pd.DataFrame([
 {"split":"FIT","H1_mean":fit_ep.H1_slope.mean(),"H2_mean":fit_ep.H2_slope.mean(),"H3_abs_mean":fit_ep.H3_abs_slope.mean(),"H3_signed_mean":fit_ep.H3_signed_slope.mean()},
 {"split":"VALIDATION","H1_mean":val_ep.H1_slope.mean(),"H2_mean":val_ep.H2_slope.mean(),"H3_abs_mean":val_ep.H3_abs_slope.mean(),"H3_signed_mean":val_ep.H3_signed_slope.mean()}])
display(aggregate); aggregate.to_csv(OUT/"v018_aggregate_endpoints.csv",index=False)


In [ ]:

def regime(df,eps):
    h=df.H3_abs_slope.to_numpy(float)
    return {"epsilon":eps,"stable_or_null_fraction":float((np.abs(h)<=eps).mean()),"amplifying_fraction":float((h>eps).mean()),"reversal_fraction":float((h < -eps).mean()),"exact_zero_fraction":float((h==0).mean())}
rows=[]
for name,d in [("FIT",fit_ep),("VALIDATION",val_ep)]:
    for e in EPS_SWEEP: rows.append({"split":name,**regime(d,e)})
regimes=pd.DataFrame(rows); display(regimes); regimes.to_csv(OUT/"v018_regime_threshold_sensitivity.csv",index=False)


In [ ]:

dose=[]
for name,d in [("FIT",fit_ep),("VALIDATION",val_ep)]:
    for a,g in d.groupby("alpha"): dose.append({"split":name,"alpha":float(a),"amplifying_fraction":float((g.H3_abs_slope>EPS_PRIMARY).mean()),"mean_H3_abs_slope":float(g.H3_abs_slope.mean())})
dose=pd.DataFrame(dose).sort_values(["split","alpha"]); display(dose); dose.to_csv(OUT/"v018_alpha_dose_response.csv",index=False)


In [ ]:

def cfg_risk(d):
    return d.assign(amplifying=d.H3_abs_slope>EPS_PRIMARY).groupby(["method","alpha","k","temperature","config_key"],dropna=False,as_index=False).agg(amplification_fraction=("amplifying","mean"),mean_H3=("H3_abs_slope","mean"))
f=cfg_risk(fit_ep); v=cfg_risk(val_ep); cfg=f.merge(v,on=["method","alpha","k","temperature","config_key"],suffixes=("_fit","_val"),validate="one_to_one")
p=pearsonr(cfg.amplification_fraction_fit,cfg.amplification_fraction_val); s=spearmanr(cfg.amplification_fraction_fit,cfg.amplification_fraction_val)
cfg["fit_centered"]=cfg.amplification_fraction_fit-cfg.groupby("alpha").amplification_fraction_fit.transform("mean")
cfg["val_centered"]=cfg.amplification_fraction_val-cfg.groupby("alpha").amplification_fraction_val.transform("mean")
pc=pearsonr(cfg.fit_centered,cfg.val_centered); sc=spearmanr(cfg.fit_centered,cfg.val_centered)
print("Pearson",p); print("Spearman",s); print("Alpha-centered Pearson",pc); print("Alpha-centered Spearman",sc)
cfg.to_csv(OUT/"v018_configuration_reproducibility.csv",index=False)


In [ ]:

val_agg=aggregate[aggregate.split=="VALIDATION"].iloc[0]
prim=regimes[(regimes.split=="VALIDATION") & np.isclose(regimes.epsilon,EPS_PRIMARY)].iloc[0]
vd=dose[dose.split=="VALIDATION"].sort_values("alpha")
criteria={
 "H1_positive":bool(val_agg.H1_mean>0),"H2_positive":bool(val_agg.H2_mean>0),"H3_positive":bool(val_agg.H3_abs_mean>0),
 "stable_majority":bool(prim.stable_or_null_fraction>0.5),"amplification_minority":bool(prim.amplifying_fraction<0.5),
 "alpha_end_to_end_increase":bool(vd.iloc[-1].amplifying_fraction>vd.iloc[0].amplifying_fraction)}
n=sum(criteria.values()); CLAIM_GATE="REPLICATION" if n==len(criteria) else ("PARTIAL_REPLICATION" if n>=len(criteria)//2 else "NON_REPLICATION")
print("="*80); print("ARC-v0.18 CLAIM GATE"); print("="*80)
for k,vv in criteria.items(): print(f"{k:35s}",vv)
print("PASS",n,"/",len(criteria)); print("CLAIM GATE:",CLAIM_GATE)


In [ ]:

report={
 "status":"ARC_V018_CROSS_ENCODER_REPLICATION_COMPLETE","protocol_sha256":PROTOCOL_SHA,"encoder":ENCODER_NAME,
 "fit_query_count":len(FIT_IDS),"validation_query_count":len(VAL_IDS),"one_shot_summary":summary.to_dict("records"),
 "aggregate_endpoints":aggregate.to_dict("records"),"primary_validation_regime":prim.to_dict(),"validation_alpha_dose_response":vd.to_dict("records"),
 "configuration_reproducibility":{"pearson_r":float(p.statistic),"spearman_rho":float(s.statistic),"alpha_centered_pearson_r":float(pc.statistic),"alpha_centered_spearman_rho":float(sc.statistic)},
 "claim_gate_criteria":criteria,"claim_gate":CLAIM_GATE,"test_accessed":False,"test_relevance_accessed":False,
 "completed_at_utc":datetime.now(timezone.utc).isoformat(),
 "constraints":["Cross-encoder-family replication, not a new primary confirmation.","SQ8 is a relative higher-fidelity comparator, not an exact oracle.","Negative/partial results are retained.","No FEVER test outcomes are used."]}
REPORT=OUT/"v018_cross_encoder_replication_report.json"; REPORT.write_text(json.dumps(report,indent=2,sort_keys=True)); RSH=sha256_file(REPORT)
(OUT/"V018_REPORT_SHA256.txt").write_text(f"{RSH}  {REPORT.name}\n")
print("="*90); print("ARC-v0.18 CROSS-ENCODER FEVER REPLICATION — COMPLETE"); print("Claim gate:",CLAIM_GATE); print("Output:",OUT); print("Report SHA-256:",RSH); print("Test accessed:",False); print("="*90)
